# Notebook 05 — Separable Linear & 2D Bilinear Image Upsampling

**Session:** Interpolation and Image upsampling  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement 2x image upsampling using **separable 1D linear interpolation** (horizontal
row pass followed by vertical column pass with boundary clamping), implement direct
**2D bilinear interpolation** over the 4 surrounding pixel corners, and compare against
`cv2.resize(..., interpolation=cv2.INTER_LINEAR)`.

In [ ]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

IMAGE_PATH = Path("../data/nature.jpg")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Separable Linear Interpolation for Image Resizing

In [ ]:
def linear_interpolation(p1: tuple[float, float], p2: tuple[float, float], x: float) -> float:
    """Linearly interpolate between p1=(x1, y1) and p2=(x2, y2)."""
    x1, y1 = p1
    x2, y2 = p2
    if x1 == x2:
        return float(y1)
    return ((x2 - x) / (x2 - x1)) * y1 + ((x - x1) / (x2 - x1)) * y2


def resize_linear_separable(
    source: list[list[int]],
    w_scale: int = 2,
    h_scale: int = 2,
) -> list[list[float]]:
    """Upsample a 2D grayscale grid using separable horizontal then vertical linear passes."""
    source_h, source_w = len(source), len(source[0])
    target_w = source_w * w_scale
    target_h = source_h * h_scale

    # Step 1: interpolate horizontally across each row (clamp x2 at source_w - 1)
    horiz = [[0.0] * target_w for _ in range(source_h)]
    for i in range(source_h):
        row = source[i]
        for j in range(target_w):
            x = j / w_scale
            x1 = math.floor(x)
            x2 = min(math.ceil(x), source_w - 1)
            horiz[i][j] = linear_interpolation((x1, row[x1]), (x2, row[x2]), x)

    # Step 2: interpolate vertically down each column (clamp y2 at source_h - 1)
    out = [[0.0] * target_w for _ in range(target_h)]
    for j in range(target_w):
        for i in range(target_h):
            y = i / h_scale
            y1 = math.floor(y)
            y2 = min(math.ceil(y), source_h - 1)
            out[i][j] = linear_interpolation((y1, horiz[y1][j]), (y2, horiz[y2][j]), y)
    return out

## 2. Direct 2D Bilinear Interpolation & Verification Against Separable Pass

In [ ]:
def resize_bilinear_2d(
    source: np.ndarray,
    target_h: int,
    target_w: int,
) -> np.ndarray:
    """Vectorized 2D bilinear interpolation matching corner-aligned coordinates."""
    src_h, src_w = source.shape
    y_coords = np.linspace(0, src_h - 1, target_h)
    x_coords = np.linspace(0, src_w - 1, target_w)
    grid_y, grid_x = np.meshgrid(y_coords, x_coords, indexing="ij")

    y0 = np.floor(grid_y).astype(np.int32)
    x0 = np.floor(grid_x).astype(np.int32)
    y1 = np.minimum(y0 + 1, src_h - 1)
    x1 = np.minimum(x0 + 1, src_w - 1)

    wy = grid_y - y0
    wx = grid_x - x0

    q11 = source[y0, x0].astype(np.float64)
    q21 = source[y0, x1].astype(np.float64)
    q12 = source[y1, x0].astype(np.float64)
    q22 = source[y1, x1].astype(np.float64)

    result = (
        q11 * (1.0 - wx) * (1.0 - wy)
        + q21 * wx * (1.0 - wy)
        + q12 * (1.0 - wx) * wy
        + q22 * wx * wy
    )
    return np.clip(np.rint(result), 0, 255).astype(np.uint8)


img_gray = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
h, w = img_gray.shape

sep_linear = np.clip(np.rint(resize_linear_separable(img_gray.tolist(), 2, 2)), 0, 255).astype(
    np.uint8
)
bilinear_2d = resize_bilinear_2d(img_gray, h * 2, w * 2)
cv_bilinear = cv2.resize(img_gray, (w * 2, h * 2), interpolation=cv2.INTER_LINEAR)

cv2.imwrite(str(OUTPUT_DIR / "05_nature_linear_2x.jpg"), sep_linear)
cv2.imwrite(str(OUTPUT_DIR / "05_nature_bilinear_2x.jpg"), bilinear_2d)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(sep_linear, cmap="gray")
axes[0].set_title(f"Separable 1D Linear 2x {sep_linear.shape}")
axes[0].axis("off")

axes[1].imshow(cv_bilinear, cmap="gray")
axes[1].set_title(f"OpenCV INTER_LINEAR 2x {cv_bilinear.shape}")
axes[1].axis("off")

plt.tight_layout()
plt.show()